# RAG 第 8 课：Eval Set 与 Error Analysis

前两课我们已经学了：
- 怎么评估检索阶段
- 怎么评估生成阶段

但真实项目里，真正困难的地方往往不是“怎么写指标函数”，而是：

- 评测集该怎么构建？
- 什么样的 query 才算覆盖全面？
- 分数下降时，问题到底出在哪一类样本上？

这节课我们就来做一个教学版的 `Eval Set + Error Analysis`。

In [ ]:
# 先清理 GPU 缓存
import gc

try:
    import torch
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('GPU cache cleared.')
    else:
        print('CUDA not available, skip GPU cache clear.')
except Exception as e:
    print(f'Torch not available, skip GPU cache clear. ({e})')

## 1. 为什么 eval set 设计很重要

如果你的评测集只包含最简单的 query，比如：
- “E401 是什么？”
- “怎么重置密码？”

那系统看起来可能分数很高，但一上线就会暴露问题。

真实项目里，query 往往会分成很多类型：
- 直接问法
- 同义改写
- 带错误码 / 产品名 / 版本号
- 多跳问题
- 含糊问题
- 超出知识库的问题

所以一个好的 eval set，核心不是“多”，而是：

**覆盖关键难点。**

## 2. 一个教学版知识库

我们继续用一个小知识库，但这次会故意让 query 类型更多样一点。

In [ ]:
import re
from collections import Counter, defaultdict

kb = [
    'E401 means authentication failed because the API token is invalid or expired.',
    'Reset your password by opening Settings, then Security, then Reset Password.',
    'Chunking splits long documents into smaller passages to improve retrieval focus.',
    'Embeddings convert text into vectors so semantic similarity can be measured.',
    'Hybrid search combines lexical matching and dense retrieval scores.',
    'Reranking reorders retrieved candidates so the strongest evidence appears first.',
    'Billing reports can be exported from the Reports page as CSV files.',
]

for i, doc in enumerate(kb):
    print(f'doc {i}: {doc}')

## 3. 构造一个更像真实项目的 eval set

这里我们给每条样本都加一个 `category`。
后面做 error analysis 时，这个字段非常重要。

真实工作里，你经常会按这些维度切分：
- 问题类型
- 产品线
- 难度等级
- 是否多跳
- 是否越界问题

In [ ]:
eval_set = [
    {
        'query': 'What does E401 mean?',
        'gold_doc_id': 0,
        'category': 'direct_fact',
    },
    {
        'query': 'Why am I seeing authentication failed with E401?',
        'gold_doc_id': 0,
        'category': 'paraphrase',
    },
    {
        'query': 'How can I change my password?',
        'gold_doc_id': 1,
        'category': 'paraphrase',
    },
    {
        'query': 'Why do RAG systems use chunking?',
        'gold_doc_id': 2,
        'category': 'concept',
    },
    {
        'query': 'What are embeddings for?',
        'gold_doc_id': 3,
        'category': 'short_query',
    },
    {
        'query': 'Why not use only dense retrieval? Why add hybrid search?',
        'gold_doc_id': 4,
        'category': 'concept',
    },
    {
        'query': 'Why do we rerank after retrieval?',
        'gold_doc_id': 5,
        'category': 'concept',
    },
    {
        'query': 'How do I export billing reports?',
        'gold_doc_id': 6,
        'category': 'product_howto',
    },
    {
        'query': 'Can I download reports as CSV?',
        'gold_doc_id': 6,
        'category': 'paraphrase',
    },
]

for item in eval_set:
    print(item)

## 4. 一个教学版检索器

这里我们继续用一个简单检索器，故意让它不是特别强。
这样后面的 error analysis 才更有东西可看。

In [ ]:
def normalize_token(token):
    token = token.lower()
    token = re.sub(r'[^a-z0-9]+', '', token)
    if token.endswith('ing') and len(token) > 5:
        token = token[:-3]
    elif token.endswith('ed') and len(token) > 4:
        token = token[:-2]
    elif token.endswith('s') and len(token) > 4:
        token = token[:-1]
    return token


def tokenize(text):
    return [t for t in (normalize_token(x) for x in text.split()) if t]


def lexical_score(query, text):
    q_tokens = tokenize(query)
    d_tokens = set(tokenize(text))
    score = 0.0
    for token in q_tokens:
        if token in d_tokens:
            score += 1.0
            if any(ch.isdigit() for ch in token):
                score += 1.5
    return score


def rank_docs(query):
    rows = []
    for doc_id, doc in enumerate(kb):
        rows.append({'doc_id': doc_id, 'score': lexical_score(query, doc), 'text': doc})
    return sorted(rows, key=lambda x: x['score'], reverse=True)


example_query = eval_set[1]['query']
print('query:', example_query)
for row in rank_docs(example_query)[:5]:
    print(f"doc={row['doc_id']} | score={row['score']:.3f}")
    print(row['text'])
    print()

## 5. 先算整体指标

这一步和上一课类似，但现在我们要开始准备“分组分析”的基础数据。

In [ ]:
def hit_at_k(ranked_ids, gold_doc_id, k):
    return 1.0 if gold_doc_id in ranked_ids[:k] else 0.0


def reciprocal_rank(ranked_ids, gold_doc_id):
    for idx, doc_id in enumerate(ranked_ids, start=1):
        if doc_id == gold_doc_id:
            return 1.0 / idx
    return 0.0


def evaluate(eval_set, k=3):
    rows = []
    hit_scores = []
    rr_scores = []

    for item in eval_set:
        ranked = rank_docs(item['query'])
        ranked_ids = [row['doc_id'] for row in ranked]
        hit = hit_at_k(ranked_ids, item['gold_doc_id'], k)
        rr = reciprocal_rank(ranked_ids, item['gold_doc_id'])

        rows.append({
            'query': item['query'],
            'category': item['category'],
            'gold_doc_id': item['gold_doc_id'],
            'top_3': ranked_ids[:3],
            'hit@3': hit,
            'rr': rr,
        })
        hit_scores.append(hit)
        rr_scores.append(rr)

    summary = {
        'Hit@3': sum(hit_scores) / len(hit_scores),
        'MRR': sum(rr_scores) / len(rr_scores),
    }
    return summary, rows

In [ ]:
summary, rows = evaluate(eval_set, k=3)
print(summary)

## 6. Error Analysis 的第一步：看失败样本

真实项目里，不要只盯着总分。
你更需要先回答：

**哪些 query 失败了？**

In [ ]:
failures = [row for row in rows if row['hit@3'] == 0.0]
print('失败样本数:', len(failures))
for row in failures:
    print('query:', row['query'])
    print('category:', row['category'])
    print('gold_doc_id:', row['gold_doc_id'])
    print('top_3:', row['top_3'])
    print('-' * 80)

## 7. Error Analysis 的第二步：按 category 分组

这一步特别关键。
你会发现整体分数还可以，但某一类样本可能非常差。

这正是为什么 eval set 里要有人为标注的 `category`。

In [ ]:
def summarize_by_category(rows):
    grouped = defaultdict(list)
    for row in rows:
        grouped[row['category']].append(row)

    category_summary = []
    for category, items in grouped.items():
        hit_avg = sum(item['hit@3'] for item in items) / len(items)
        mrr_avg = sum(item['rr'] for item in items) / len(items)
        category_summary.append({
            'category': category,
            'count': len(items),
            'Hit@3': hit_avg,
            'MRR': mrr_avg,
        })
    return sorted(category_summary, key=lambda x: (x['Hit@3'], x['MRR']))


category_summary = summarize_by_category(rows)
for item in category_summary:
    print(item)

## 8. Error Analysis 的第三步：猜原因

到这一步，真正有价值的工作才开始。

比如你可能会发现：
- `direct_fact` 很强
- `paraphrase` 明显更差
- `short_query` 容易歧义
- `concept` 类问题更依赖语义检索

那下一步可能的改进方向就会很清楚：
- 补 synonym / query rewrite
- 上 embedding 检索
- 加 reranking
- 单独补某类 query 的训练数据或测试样本

In [ ]:
for row in rows:
    if row['rr'] < 1.0:
        print('需要关注的样本:')
        print('query:', row['query'])
        print('category:', row['category'])
        print('gold_doc_id:', row['gold_doc_id'])
        print('top_3:', row['top_3'])
        print()

## 9. 一个很实用的经验

真实项目里，eval set 往往不是一次性做完，而是不断迭代：

1. 先做一个小而有代表性的初版
2. 跑系统，看失败样本
3. 把真实失败 case 回填进 eval set
4. 再继续优化系统

所以 eval set 不是静态资产，更像是一份不断进化的“问题地图”。

## 10. 本课小结

这节课你要带走的核心是：

1. 评估不只是算分，更要设计好 eval set。
2. Eval set 里最好给样本打标签，比如 `category`。
3. Error analysis 最重要的是：找到哪一类问题最差。
4. 一个好的 eval set 会随着系统迭代不断扩充。

下一课最自然的衔接就是：
**用更真实的方式构建一个完整 RAG pipeline：query rewrite + retrieve + rerank + generate。**